## TODO List

- [ ] Compare the Multi-Layer Perceptron (MLP) with two variants of Convolutional Neural Networks (CNNs).

- [ ] Investigate what happens when a shape appears at a position that was not seen during training.

- [ ] Investigate how shuffling the pixels of all images affects model performance and classification accuracy.
- [] Replace the pooling by dilated convolutions and find out when pooling and when
dilation is the right choice.

In [1]:
import numpy as np
import torch

IMG = 28

SHAPE_NAMES = ["square", "ring", "plus", "cross"]


def draw_shape(kind, size):
    """
    Template (size x size) of the shape:
    0 = square, 1 = ring, 2 = plus, 3 = cross.
    """

    m = np.zeros((size, size), dtype=np.float32)
    c = (size - 1) / 2

    yy, xx = np.mgrid[:size, :size]

    if kind == 0:
        m[:] = 1.0

    elif kind == 1:
        r = np.hypot(yy - c, xx - c)
        m[(r <= size / 2) & (r >= size / 2 - 1.6)] = 1.0

    elif kind == 2:
        m[size // 2 - 1:size // 2 + 1, :] = 1.0
        m[:, size // 2 - 1:size // 2 + 1] = 1.0

    else:
        d = np.arange(size)
        m[d, d] = 1.0
        m[d, size - 1 - d] = 1.0
        m = np.maximum(m, np.roll(m, 1, axis=1))

    return m


def make_shapes(n_samples, size=9, noise=0.3, pos_range=None, seed=0):
    """
    Return images X (n, 1, 28, 28), float32,
    and labels y (n,), int64.

    The row and column of the upper-left corner of the shape
    are drawn uniformly from pos_range = (lo, hi).

    Default: (0, 28 - size), meaning anywhere.
    """

    lo, hi = pos_range if pos_range is not None else (0, IMG - size)

    rng = np.random.default_rng(seed)

    templates = [draw_shape(k, size) for k in range(4)]

    y = rng.integers(0, 4, n_samples)

    pos = rng.integers(lo, hi + 1, (n_samples, 2))

    X = np.zeros((n_samples, 1, IMG, IMG), dtype=np.float32)

    for i in range(n_samples):
        r, c = pos[i]
        X[i, 0, r:r + size, c:c + size] = templates[y[i]]

    X += noise * rng.standard_normal(X.shape).astype(np.float32)

    return (
        torch.from_numpy(X),
        torch.from_numpy(y.astype(np.int64))
    )

In [2]:
# Generate 2000 test images
X_test, y_test = make_shapes(2000, seed=99)

# Generate training datasets of different sizes
N_values = [100, 500, 1000, 2000, 5000]

training_sets = {}

for N in N_values:
    X_train, y_train = make_shapes(N, seed=100)
    training_sets[N] = (X_train, y_train)